# XGBoost 100 m prediction-smoothing verification

This notebook is a report front-end for the reusable production pipeline. It compares fold-safe Gaussian prediction smoothing at σ = 0, 15, 30, 45 and 60 m on the corrected mutually-exclusive-ground plus overlapping-canopy contract. Observed Landsat temperatures are never smoothed.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from greenwave_local_layers.image_regression_smoothing_benchmark import (
    BENCHMARK_PREDICTIONS_PATH, BENCHMARK_REPORT_PATH, REPORT_PATH, run_smoothing_benchmark,
)

## Run or reuse the complete benchmark

In [ ]:
if not BENCHMARK_REPORT_PATH.exists():
    run_smoothing_benchmark(device="cuda")
report = json.loads(BENCHMARK_REPORT_PATH.read_text(encoding="utf-8"))
production = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
report["pairedBootstrap"], report["selectedSigmaByOuterFold"], report["productionSigmaMeters"]

## Held-out metrics and selected smoothing

In [ ]:
{
    "raw": report["pooledRawMetrics"],
    "smoothing-aware": report["pooledSmoothedMetrics"],
    "smoothed minus raw": report.get("smoothedMinusRaw"),
    "paired sector bootstrap": report["pairedBootstrap"],
}

## Fold choices and production contract

In [ ]:
fold_summary = [{
    "fold": fold["fold"],
    "selected_sigma_m": fold["selectedSigmaMeters"],
    "raw_metrics": fold["rawMetrics"],
    "smoothed_metrics": fold["smoothedMetrics"],
    "retained_features": fold["smoothingAwarePipeline"]["retainedFeatures"],
    "parameters": fold["smoothingAwarePipeline"]["parameters"],
} for fold in report["outerFolds"]]
fold_summary

In [ ]:
{
    "production_sigma_m": production["final"]["smoothingSigmaMeters"],
    "smoothing_promoted": production["final"]["smoothingPromoted"],
    "parameters": production["final"]["parameters"],
    "retained_features": production["final"]["retainedFeatures"],
    "rejected_features": production["final"]["rejectedFeatures"],
    "model_sha256": production["final"]["modelSha256"],
}

## Observed versus predicted and residual diagnostics

In [ ]:
with np.load(BENCHMARK_PREDICTIONS_PATH, allow_pickle=False) as values:
    observed = values["observed_c"]
    raw = values["raw_predicted_c"]
    smoothed = values["smoothed_predicted_c"]

figure, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(observed, raw, s=3, alpha=.15, label="Unsmoothed")
axes[0].scatter(observed, smoothed, s=3, alpha=.15, label="Smoothing-aware")
limits = [float(observed.min()), float(observed.max())]
axes[0].plot(limits, limits, color="black", linewidth=1)
axes[0].set(xlabel="Observed LST (°C)", ylabel="Predicted LST (°C)")
axes[0].legend()
axes[1].hist(raw - observed, bins=60, alpha=.55, label="Unsmoothed")
axes[1].hist(smoothed - observed, bins=60, alpha=.55, label="Smoothing-aware")
axes[1].set(xlabel="Prediction residual (°C)", ylabel="Observations")
axes[1].legend()
figure.tight_layout()